# Task #2: Reshape QPASS observations and check data quality

This notebook now uses Dylan's task #1 loader for all workbooks in `data/raw/QPASS/`, then calls the task #2 functions in `src/qpass_observations.py`.

Run cells in order. If the raw folder is empty, first run `python -m src.download_qpass` from the repository root. Original workbooks are never edited. Loader exclusions are retained in metadata; this audit covers rows returned by the loader.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.qpass_loader import load_all_qpass
from src.qpass_observations import (
    reshape_loaded_observations,
    validate_observations,
)

RAW_DIR = REPO_ROOT / "data" / "raw" / "QPASS"
OUTPUT_DIR = REPO_ROOT / "data" / "processed" / "qpass-notebook"


## 1. Load the supported workbooks

Dylan's loader combines the supported workbook formats into one DataFrame, preserving source file, sheet, row, academic year, and quarter. Counts remain in `open_8am`, `open_10am`, `open_12pm`, and `open_2pm`. Display its audit information as well: rows excluded by the loader do not reach task #2's checks.


In [ ]:
loaded_data = load_all_qpass(RAW_DIR)
loader_audit = loaded_data.attrs.copy()
print("Workbooks loaded:", len(loader_audit["files"]))
print("Source rows returned by loader:", len(loaded_data))
display(loaded_data.head())
print("Loader exclusions (per file and sheet):")
display(loader_audit["dropped_rows"])


## 2. Reshape and standardize values

`melt` turns four time columns into four observation rows per source row. We preserve source values in `raw_*` columns before converting dates, trimming identity strings, and parsing counts. Unparseable counts become missing values and will be flagged. Excel row numbers provide a way to find questionable values in the source workbook. Dates are campus local wall time; workbook capacity still needs inventory verification.


In [ ]:
# These functions are imported from src/qpass_observations.py above.
print("Reshaping function comes from:", reshape_loaded_observations.__module__)


## 3. Define the quality checks

Checks cover missing required values, negative or nonfinite counts, fractional counts, zero capacity, open spaces above capacity, and duplicate identities. Every member of a duplicate group is flagged. Counts from different files should be concatenated before validation to catch cross-file duplicates.

`quality_flag` lists all failed checks, separated by `|`. `quality_pass` is true only when no checks fail. Failure counts overlap; use the `any_check` summary row for the number of unique failed records.


The validation function is imported above rather than redefined here. Open `src/qpass_observations.py` to read its implementation.

`validate_observations(observations)` returns two DataFrames: the observations with flags, and per-check pass/fail counts. Tests in `tests/test_qpass_observations.py` call this same function with small sample tables containing known problems.


## 4. Run reshaping and validation

The adapter reshapes every row returned by Dylan's loader into four time observations. Missing counts and partially filled observations are retained and flagged. Rows excluded by the loader are recorded separately in its audit metadata.


The connected workflow is:

```text
Excel workbooks -> load_all_qpass() [Dylan, src/qpass_loader.py]
               -> reshape_loaded_observations() [Vivaan, src/qpass_observations.py]
               -> validate_observations() -> results and exports
```

The adapter maps Dylan's headers to the reshaping function's expected headers. It retains each file's metadata, structure levels, subtype fields, and loader audit attributes. Missing report categories remain missing for team review; no category mapping is guessed.


In [ ]:
observations = reshape_loaded_observations(loaded_data)
audited, quality_summary = validate_observations(observations)
questionable_records = audited.loc[~audited["quality_pass"]].copy()

assert len(observations) == 4 * len(loaded_data)
assert len(audited) == len(observations)
print(f"Observation rows: {len(audited):,}")
print(f"Passing rows: {audited['quality_pass'].sum():,}")
print(f"Flagged rows: {len(questionable_records):,}")
display(quality_summary)
display(audited[["source_file", "source_sheet", "source_row", "observation_datetime",
                 "facility_name", "parking_category", "parking_subtype",
                 "capacity", "open_spaces", "quality_flag"]].head(12))


## 5. Examine questionable records

These records need review, not automatic deletion. Use the source file, sheet, and row to investigate the original values. The sample below displays up to five records per failed check; the full dataframe remains available as `questionable_records`.


In [ ]:
audit_columns = ["source_file", "source_sheet", "source_row", "observation_datetime",
                 "facility_name", "parking_category", "parking_subtype",
                 "raw_capacity", "raw_open_spaces", "quality_flag"]
for check in quality_summary.loc[quality_summary["failed"].gt(0), "check"]:
    if check == "any_check":
        continue
    print(check)
    matching = audited["quality_flag"].str.split("|").map(lambda flags: check in flags)
    display(audited.loc[matching, audit_columns].head())


## 6. Export the deliverables

This cell writes generated CSV reports locally under `data/processed/qpass-notebook`. It does not edit the workbook or publish to GitHub. CSV readers should parse `observation_datetime` and `observation_date` as dates when reloading.


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
audited.to_csv(OUTPUT_DIR / "observations.csv", index=False)
questionable_records.to_csv(OUTPUT_DIR / "questionable_records.csv", index=False)
quality_summary.to_csv(OUTPUT_DIR / "quality_summary.csv", index=False)
run_metadata = {
    "source_files": loader_audit["files"],
    "loader_audit": loader_audit,
    "observation_rows": len(audited),
    "passing_rows": int(audited["quality_pass"].sum()),
    "flagged_rows": len(questionable_records),
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")
print("Reports saved to:", OUTPUT_DIR.resolve())


## Quality-flag definitions and unresolved questions

| Flag | Meaning |
| --- | --- |
| `missing_<field>` | Blank, missing, or unparseable required value |
| `negative_<count>` | Count below zero |
| `nonfinite_<count>` | Infinite numeric value |
| `fractional_<count>` | Finite count is not an integer |
| `zero_capacity` | Zero stalls; availability fraction cannot use this denominator |
| `open_exceeds_capacity` | Open-space count is larger than workbook capacity |
| `duplicate_observation` | Repeated date/time/facility/category/subtype identity |
| `pass` | No checks failed |

- Confirm output column names with the other data tasks.
- Investigate duplicate identities: a more specific location key may be needed.
- Missing open-space values must not be treated as zero.
- Confirm handling of zero-capacity allocations and historical inventory matches.
- The general loader now handles the older weekly-sheet formats. Review its recorded exclusions alongside these quality flags.
- Structure levels and optional subtype fields are included in duplicate detection when present. Missing report categories in older formats remain flagged for category reconciliation.
- Passing these checks does not establish permit eligibility or forecast reliability outside surveyed dates.

The summary above is calculated from all workbooks currently in `RAW_DIR`; it replaces the earlier single-workbook demonstration. Counts depend on the downloaded files and the loader's exclusions.


## Combined audit results - 2026-10-18

This saved report was calculated using the task #1 loader and the task #2 reshape/validation functions. It remains visible when notebook outputs are cleared. Rerun the pipeline and refresh this report if the input files or checks change.

- Workbooks loaded: **15**.
- Source rows returned by the loader: **23,428**.
- Reshaped observation rows: **93,712** (four per loader row).
- Passed every implemented check: **39,623**.
- Flagged for review: **54,089**.

Flagged records remain in the full observations table. Passing means only that these checks passed; capacity, category, permit, and geographic verification are separate steps.

| Check | Failed | Passed |
| --- | ---: | ---: |
| `missing_observation_datetime` | 0 | 93,712 |
| `missing_facility_name` | 0 | 93,712 |
| `missing_parking_category` | 40,624 | 53,088 |
| `missing_parking_subtype` | 252 | 93,460 |
| `missing_capacity` | 72 | 93,640 |
| `missing_open_spaces` | 5,381 | 88,331 |
| `negative_capacity` | 0 | 93,712 |
| `nonfinite_capacity` | 0 | 93,712 |
| `fractional_capacity` | 0 | 93,712 |
| `negative_open_spaces` | 1 | 93,711 |
| `nonfinite_open_spaces` | 0 | 93,712 |
| `fractional_open_spaces` | 60 | 93,652 |
| `zero_capacity` | 8,052 | 85,660 |
| `open_exceeds_capacity` | 5,442 | 88,270 |
| `duplicate_observation` | 3,344 | 90,368 |
| `any_check` | 54,089 | 39,623 |

Failures overlap: one observation can fail several checks. `any_check` counts unique failed rows.

### Examples requiring review

These are actual output records. The explanations describe why the checks fired, not verified corrections to the source. File, sheet, and Excel row identify where to investigate.

**`missing_parking_category`**

- Source: `QPASS 2018-2019-1 Summer.xlsx` -> `1 Summer 2018 June 18` -> Excel row **5**.
- Observation: 2018-06-18 08:00:00; facility **Athena**; category **(missing)**; subtype **Visitor**.
- Original capacity: `64`; original open count: `0`.
- Why flagged: The loader provided no Report Type category. Keep it missing until the category-reconciliation task confirms a mapping.

**`missing_parking_subtype`**

- Source: `QPASS 2018-2019-1 Summer.xlsx` -> `3 Summer 2018 July 2nd` -> Excel row **65**.
- Observation: 2018-07-02 08:00:00; facility **P017**; category **(missing)**; subtype **(missing)**.
- Original capacity: `258`; original open count: `(missing)`.
- Why flagged: The source Type is missing or blank after trimming. Do not guess an allocation.

**`missing_capacity`**

- Source: `QPASS 2024-2025-1  Summer.xlsm` -> `QPASS Data Entry` -> Excel row **107**.
- Observation: 2024-07-01 08:00:00; facility **Campus Point East**; category **(missing)**; subtype **(missing)**.
- Original capacity: `(missing)`; original open count: `(missing)`.
- Why flagged: Capacity is missing or could not be parsed as a number. An availability fraction cannot yet be calculated.

**`missing_open_spaces`**

- Source: `QPASS 2018-2019-1 Summer.xlsx` -> `1 Summer 2018 June 18` -> Excel row **43**.
- Observation: 2018-06-18 08:00:00; facility **Pangea**; category **(missing)**; subtype **B**.
- Original capacity: `159`; original open count: `(missing)`.
- Why flagged: The count is missing or could not be parsed. Do not interpret it as zero available spaces.

**`negative_open_spaces`**

- Source: `QPASS 2018-2019-3 Winter.xlsx` -> `W19 W1 Data` -> Excel row **61**.
- Observation: 2019-01-11 10:00:00; facility **Osler**; category **(missing)**; subtype **B**.
- Original capacity: `552`; original open count: `-3.064215547965432e-14`.
- Why flagged: The strict check flags this value because it is below zero. Its magnitude is extremely small and may reflect floating-point rounding in spreadsheet calculations; review the source before deciding whether to treat it as zero. The original value is preserved and no rounding tolerance is applied.

**`fractional_open_spaces`**

- Source: `QPASS 2018-2019-3 Winter.xlsx` -> `W19 W1 Data` -> Excel row **18**.
- Observation: 2019-01-07 08:00:00; facility **Osler**; category **(missing)**; subtype **A**.
- Original capacity: `156`; original open count: `109.19999999999999`.
- Why flagged: The reported open-space count is fractional; review whether this is a source error or a value with a different meaning.

**`zero_capacity`**

- Source: `QPASS 2024-2025-2 Fall.xlsm` -> `QPASS Data Entry (Week 0)` -> Excel row **90**.
- Observation: 2024-09-26 08:00:00; facility **P602**; category **A**; subtype **2 hour A**.
- Original capacity: `0`; original open count: `0`.
- Why flagged: Capacity is zero. Do not divide by this value when calculating availability.

**`open_exceeds_capacity`**

- Source: `QPASS 2018-2019-1 Summer.xlsx` -> `1 Summer 2018 June 18` -> Excel row **45**.
- Observation: 2018-06-18 08:00:00; facility **Pangea**; category **(missing)**; subtype **B**.
- Original capacity: `77`; original open count: `102`.
- Why flagged: The open-space count exceeds the workbook capacity; check the counts and historical inventory.

**`duplicate_observation`**

- Source: `QPASS 2024-2025-2 Fall.xlsm` -> `QPASS Data Entry (Week 0)` -> Excel row **110**.
- Observation: 2024-09-26 08:00:00; facility **Theatre District Garage**; category **S**; subtype **S**.
- Original capacity: `50`; original open count: `(missing)`.
- Why flagged: The date/time, facility, category, subtype, and available level/subtype fields repeat. Determine whether they are duplicates or require a more specific allocation identifier.

### Assumptions and unresolved questions

- These checks cover only rows returned by the loader. Its skipped sheets and dropped rows must also be reviewed; they are displayed earlier and included in exported run metadata.
- Confirm the output schema with the team. In particular, missing Report Type values in older formats are flagged rather than inferred from Type.
- Resolve repeated identities before deduplication. Level and optional Sub Type are included when available, but other allocation details may still be needed.
- Investigate missing counts and fractional or negative values against source workbooks. No corrections or imputation have been applied.
- Workbook capacity has not yet been matched to the historical parking inventory. Zero or inconsistent capacities need capacity-task review.
- This is a quality audit of historical survey observations, not verification of live parking availability.

### Reproduce these results

1. Install `requirements.txt` and download the supported files with `python -m src.download_qpass`.
2. Restart the notebook kernel and run all cells. The notebook calls `load_all_qpass()`, `reshape_loaded_observations()`, and `validate_observations()`.
3. The export cell writes full observations, questionable records, quality summary, and loader/run metadata under `data/processed/qpass-notebook/`. Those generated files are ignored by Git.
4. Run `python -m pytest` from the repository root to check the code.
